In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [8]:
DATA_PATH = "../data"

ratings = pd.read_csv(os.path.join(DATA_PATH, "ratings.csv"), sep=",")
movies = pd.read_csv(os.path.join(DATA_PATH, "movies.csv"), sep=",")

print(ratings.shape)
print(movies.shape)
ratings.head()

(32000204, 4)
(87585, 3)


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [5]:
movies.head(20)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [10]:
ratings.describe(include="all")

,userId,movieId,rating,timestamp
count,3.200020e+07,3.200020e+07,3.200020e+07,3.200020e+07
mean,1.002785e+05,2.931861e+04,3.540396e+00,1.275241e+09
std,5.794905e+04,5.095816e+04,1.058986e+00,2.561630e+08
min,1.000000e+00,1.000000e+00,5.000000e-01,7.896520e+08
25%,5.005300e+04,1.233000e+03,3.000000e+00,1.051012e+09
50%,1.002970e+05,3.452000e+03,3.500000e+00,1.272622e+09
75%,1.504510e+05,4.419900e+04,4.000000e+00,1.503158e+09
max,2.009480e+05,2.927570e+05,5.000000e+00,1.697164e+09


In [ ]:
ratings.info()
print("\nBraki w kolumnach:\n", ratings.isna().sum())
print("\nDuplikaty w ratings:", ratings.duplicated().sum())

In [ ]:
movies.info()
print("\nBraki w kolumnach:\n", movies.isna().sum())
print("\nDuplikaty w movies:", movies.duplicated().sum())

In [ ]:
import seaborn as sns
plt.figure(figsize=(15, 5))
sns.histplot(ratings["rating"], bins=100, kde=True)
plt.title("Rozkład ocen (rating)")

In [ ]:
print(ratings["userId"].nunique())
print(ratings["movieId"].nunique())

In [ ]:
# sprawdzamy gęstość macierzy i wartośći puste
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
n_ratings = len(ratings)

sparsity = 1 - (n_ratings / (n_users * n_movies))
print(f"Sparsity: {sparsity:.6f}")
print(f"Density: {(1 - sparsity):.6f}")

In [ ]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user_mapper = {user_id: i for i, user_id in enumerate(user_ids)}
movie_mapper = {movie_id: i for i, movie_id in enumerate(movie_ids)}

In [ ]:
user_index = ratings["userId"].map(user_mapper)
movie_index = ratings["movieId"].map(movie_mapper)

In [ ]:
import scipy.sparse as sp
csr_user_item = sp.csr_matrix((ratings["rating"], (user_index, movie_index)), shape=(n_users, n_movies))
csr_user_item.shape

In [ ]:
csr_item_user = csr_user_item.T

In [ ]:
csr_user_item.nnz

In [ ]:
# csr_test.shape # liczba wierszy i kolumn
# csr_test.nnz # liczba niezerowych elementów
# csr_test.T # transpozycja macierzy
# csr_test.toarray() # konwersja do gęstej macierzy (niezalecane dla dużych zbiorów danych)
# csr_test.getrow(0) # pobranie wiersza o indeksie 0
# csr_test.getcol(0) # pobranie kolumny o indeksie 0

In [ ]:
ratings_sorted = ratings.sort_values(by=["userId", "timestamp"], ascending=True)

In [ ]:
ratings_sorted.head(10)

In [ ]:
ratings_test = ratings_sorted.groupby("userId").tail(1)
ratings_test

In [ ]:
ratings_train = ratings_sorted.drop(ratings_test.index)
ratings_train

In [ ]:
ratings_test["userId"].unique().shape

In [ ]:
ratings_train["userId"].unique().shape

In [ ]:
ratings_test["movieId"].unique().shape

In [ ]:
ratings_train["movieId"].unique().shape

In [ ]:
set_test_movies = set(ratings_test["movieId"])
set_train_movies = set(ratings_train["movieId"])

In [ ]:
compare_movies = set_test_movies - set_train_movies
len(compare_movies)

In [ ]:
train_movies = set(ratings_train["movieId"])
ratings_test = ratings_test[ratings_test["movieId"].isin(train_movies)]

In [ ]:
ratings_sorted.describe(include="all")

In [ ]:
ratings_per_user = ratings.groupby("userId").size()
ratings_per_user

In [ ]:
ratings_per_user.min()

In [ ]:
print("Users in train:", ratings_train["userId"].nunique())
print("Users in test after filtering:", ratings_test["userId"].nunique())
print("Rows in test after filtering:", len(ratings_test))
print("Min test rows per user:", ratings_test.groupby("userId").size().min())
print("Max test rows per user:", ratings_test.groupby("userId").size().max())

In [ ]:
train_users = set(ratings_train["userId"])
test_users = set(ratings_test["userId"])

missing_test_users = train_users - test_users
len(missing_test_users)